- https://www.notion.so/verl-reTool-recipe-2398b5b7feba80a58156fa936f9f8de6
- update
    - https://github.com/zhaochenyang20/Awesome-ML-SYS-Tutorial/blob/main/rlhf/verl/multi-turn/release_log/latest_sglang.md
- 12月1日
    - 96e7071de
        - https://github.com/volcengine/verl/tree/96e7071de1bc6a7c0e12f4999f97556da9310cc3
        - [2025/07] The ReTool recipe is fully open sourced. Blog
        - https://github.com/verl-project/verl-recipe/tree/main/retool
    - https://github.com/volcengine/verl/blob/main/examples/sglang_multiturn/run_qwen3_4b_dapo_multiturn.sh
- 局限性
    - 单一type的tool，ci（code interpreter）
    - 但事实上两大类 General tool
        - code
        - search
- 理解论文及其代码
    - 数据的来源、合成及分布
    - 算法、training pipeline
    - 评估及metrics

### interleaved generation

$$
o = [t_1 \oplus c_1 \oplus f_1 \oplus \cdots \oplus t_n]
$$
- $t_i$: 自然语言推理步骤 (Thought)
- $c_i$: 代码片段 (Action)
- $f_i$: 执行器反馈结果 (Observation)

### algorithm pipeline

```python
# Algorithm: ReTool Training Pipeline
def ReTool_Train(BaseModel, Data):
  # Phase 1: Cold-Start SFT
  D_CI = synthesize_code_traces(Data)
  Policy = SFT(BaseModel, D_CI)

  # Phase 2: RL with Tool Integration
  for epoch in range(MAX_EPOCHS):
    prompts = sample(Data)
    # Interleaved Rollout
    trajectories = []
    while not done:
      tokens = Policy.generate()
      if "</code>" in tokens:
        result = Sandbox.exec(tokens.code)
        tokens += f"<interpreter>{result}</interpreter>"
    rewards = compute_outcome_reward(trajectories)
    Policy = PPO_Update(Policy, rewards)

  return Policy
```

### cold-start sft

- Firstly, we collect data through our designed pipeline for cold-start supervised fine-tuning (SFT), which provides a robust initialization for the reinforcement learning phase.
- This teaches the model an **initial competency** in tool usage and execution result analysis.
    - provides a **robust initialization** for the reinforcement learning phase.
    - ReTool employs supervised fine-tuning to learn **when and how to invoke the code interpreter** from the aforementioned dataset $\mathcal D_{CI}$ thereby enhancing the model's capability to appropriately utilize computational tools."
- Construction of SFT dataset: (Figure 8: Template Prompt for Data Curation)
    - $\mathcal D_{init}$
    - $\mathcal D_{CI}$
        - code interpreter

### RL

- Template prompt for ReTool rollout.
- results
    - 训练前，代码主要用于简单的 Calculation。训练后，出现了 Optimization, Solution Search, Geometric Calculation 等多样化用途，证明模型学会了更高级的工具使用策略。

### 数据准备

```
huggingface-cli download JoeYing/ReTool-SFT \
    --repo-type dataset \
    --local-dir JoeYing/ReTool-SFT \
    --local-dir-use-symlinks False
    
huggingface-cli download BytedTsinghua-SIA/DAPO-Math-17k \
    --repo-type dataset \
    --local-dir BytedTsinghua-SIA/DAPO-Math-17k \
    --local-dir-use-symlinks False

huggingface-cli download Maxwell-Jia/AIME_2024 \
    --repo-type dataset \
    --local-dir Maxwell-Jia/AIME_2024 \
    --local-dir-use-symlinks False

huggingface-cli download yentinglin/aime_2025 \
    --repo-type dataset \
    --local-dir yentinglin/aime_2025 \
    --local-dir-use-symlinks False
```